In [3]:
import re
import gc
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

import spacy
import scispacy
import medspacy

notes = pd.read_csv(
    "../outputs/eda_base_dataset.csv"
)

print("Dataset shape:", notes.shape)


Dataset shape: (11061, 9)


In [4]:
nlp = spacy.load("en_core_sci_lg")

nlp.add_pipe("medspacy_context")

print("Pipeline loaded.")

print("\nPipeline components:")
print(nlp.pipe_names)

Pipeline loaded.

Pipeline components:
['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'medspacy_context']


In [5]:
def clean_note(text):

    if pd.isnull(text):
        return ""

    text = str(text)
    text = re.sub(
        r"\[\*\*.*?\*\*\]",
        " ",
        text
    )

    text = re.sub(r"__+", " ", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\r+", " ", text)
    text = re.sub(r"\t+", " ", text)
    text = re.sub(r"\s+", " ", text)

    text = text.lower().strip()

    return text

sample_raw = notes.iloc[0]["TEXT"]

sample_clean = clean_note(sample_raw)

print("\n=== RAW TEXT SAMPLE ===\n")
print(sample_raw[:1000])

print("\n=== CLEANED TEXT SAMPLE ===\n")
print(sample_clean[:1000])



=== RAW TEXT SAMPLE ===

Name:  [**Known lastname 183**], [**Known firstname **] A           Unit No:  [**Numeric Identifier 17409**]

Admission Date: [**2100-7-11**]        Discharge Date: [**2100-7-15**]

Date of Birth:  [**2062-12-19**]        Sex:  F

Service:  [**Last Name (un) **]


ADDENDUM:  The patient had to spend last night ([**2100-7-12**]) because of unable to transport in the evening and
required dialysis during the day.  This morning, she is doing
well.  She was found asleep.  On examination, her abdomen was
soft, nontender, and nondistended.  She was afebrile.  She
was tolerating oral intake.  She was complaining of nausea,
but no vomiting.  She was able to eat regular meals (a full
meal).

CONDITION ON DISCHARGE:  Same.

DISCHARGE DISPOSITION:  The patient to be discharged back to
nursing facility today.



                        [**Name6 (MD) **] [**Name8 (MD) **], [**MD Number(1) 16979**]

Dictated By:[**Last Name (NamePattern1) 10534**]
MEDQUIST36
D:  [**2100-7-13

In [6]:
print("\nCleaning notes...")

tqdm.pandas()

notes["clean_text"] = notes["TEXT"].progress_apply(clean_note)

print("Cleaning complete.")


Cleaning notes...


100%|██████████| 11061/11061 [00:05<00:00, 2115.81it/s]

Cleaning complete.


In [7]:
before = len(notes)

notes = notes[
    notes["clean_text"].str.len() > 50
].copy()

after = len(notes)

print(f"\nRemoved {before - after:,} empty/short notes.")



Removed 0 empty/short notes.


In [8]:
TARGET_LABELS = {
    "PROBLEM",
    "DISEASE",
    "CONDITION",
    "SYMPTOM"
}

def extract_symptoms(doc):
    symptoms = []
    
    for ent in doc.ents:

        label = ent.label_

        if label not in TARGET_LABELS:
            continue

        context = ent._.context_attributes

        is_negated = context.get("is_negated", False)

        is_uncertain = context.get("is_uncertain", False)

        if is_negated:
            continue

        if is_uncertain:
            continue

        text = ent.text.strip().lower()


        if len(text) < 3:
            continue

        symptoms.append(text)

    return symptoms

In [9]:
import gc

def process_chunk(text_chunk):

    results = []

    pipe = nlp.pipe(
        text_chunk,
        batch_size=16,      
        n_process=1         
    )

    for doc in pipe:

        symptoms = extract_symptoms(doc)

        results.append(symptoms)

    return results

CHUNK_SIZE = 1000

all_symptoms = []

texts = notes["clean_text"]

print("\nRunning clinical NER...\n")

for start in tqdm(range(0, len(texts), CHUNK_SIZE)):

    end = start + CHUNK_SIZE

    chunk = texts.iloc[start:end]

    chunk_results = process_chunk(chunk)

    all_symptoms.extend(chunk_results)

    # Explicit cleanup
    del chunk_results
    gc.collect()

notes["symptoms"] = all_symptoms

print("\nNER extraction complete.")



Running clinical NER...



100%|██████████| 12/12 [25:32<00:00, 127.67s/it]


NER extraction complete.
